## Notes

All candidates page does not have winner and stuff\
using constituition wise data\
but the networks tab does not have any data\
the pagee source is somewhat encoded\
so forced to use selenium for each constituition\

but to extract all urls of the constituition data\
I used requests to fetch the page source of the particular yr lok sabha page

### Extracting constituition wise links

In [11]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = Path("../raw")
LINK_DIR = BASE_DIR / "lok_sabha_constituencies"
SOURCE_DIR = BASE_DIR / "lok_sabha_source_html"

LINK_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

PAGES = {
    2024: "https://www.myneta.info/LokSabha2024/",
    2019: "https://www.myneta.info/LokSabha2019/",
    2014: "https://www.myneta.info/ls2014/",
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36"
})

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def get_page(url):
    for attempt in range(3):
        try:
            response = session.get(url, timeout=(15, 60))
            response.raise_for_status()
            return response
        except requests.RequestException:
            if attempt == 2:
                raise
            time.sleep(3 * (attempt + 1))

def parse_landing_page(year, landing_url):
    response = get_page(landing_url)
    (SOURCE_DIR / f"landing_page_{year}.html").write_bytes(response.content)

    soup = BeautifulSoup(response.content, "html.parser")
    rows = []
    seen = set()

    for a in soup.find_all("a", href=True):
        if "action=show_candidates" not in a["href"]:
            continue

        container = a.find_parent(
            "div", class_=lambda c: c and "w3-third" in c
        )
        if container is None:
            continue

        button = container.find("button")
        if button is None:
            continue

        state_ut = button.get_text(" ", strip=True)
        state_ut = re.sub(r"\s+", " ", state_ut).strip()

        all_const_link = container.find(
            "a", href=re.compile(r"action=show_constituencies")
        )
        state_id = None

        if all_const_link:
            state_params = parse_qs(urlparse(all_const_link["href"]).query)
            state_id = state_params.get("state_id", [None])[0]

        link = urljoin(landing_url, a["href"])
        params = parse_qs(urlparse(link).query)
        const_id = params.get("constituency_id", [None])[0]

        if not const_id or link in seen:
            continue
        seen.add(link)

        title = a.get("title", "")
        date_match = re.search(
            r"Date of Election\s*:?\s*(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
            title,
            re.I,
        )
        election_date = date_match.group(1) if date_match else None

        rows.append({
            "year": year,
            "state_ut": state_ut,
            "state_id": state_id,
            "const": a.get_text(" ", strip=True),
            "const_id": str(const_id),
            "link": link,
            "election_date": election_date,
        })

    return pd.DataFrame(rows, columns=[
        "year", "state_ut", "state_id", "const",
        "const_id", "link", "election_date"
    ])

link_dfs = {}

for year, url in PAGES.items():
    print(f"Scraping {year}...")
    try:
        df = parse_landing_page(year, url)
        link_dfs[year] = df
        df.to_csv(LINK_DIR / f"constituency_links_{year}.csv", index=False)
        print(f"{year}: extracted {len(df)} constituency links")
        display(df.head(5))
    except requests.RequestException as e:
        print(f"Failed to scrape {year}: {e}")

if link_dfs:
    all_links = pd.concat(link_dfs.values(), ignore_index=True)
    all_links.to_csv(
        LINK_DIR / "constituency_links_all_years.csv", index=False
    )

    print("\nState counts by year:")
    display(
        all_links.groupby(["year", "state_ut"], dropna=False)
        .size().reset_index(name="constituency_count")
    )

    print("\nMissing values:")
    display(all_links.isna().sum().rename("missing_count").to_frame())

    print("\nTarget-state rows:")
    display(
        all_links[
            all_links["state_ut"].str.upper().isin([
                "TAMIL NADU", "UTTAR PRADESH", "BIHAR", "MAHARASHTRA"
            ])
        ].groupby(["year", "state_ut"])
        .size().reset_index(name="constituency_count")
    )

Scraping 2024...
2024: extracted 546 constituency links


,year,state_ut,state_id,const,const_id,link,election_date
0,2024,ANDAMAN AND NICOBAR ISLANDS,1,ANDAMAN AND NICOBAR ISLANDS,579,https://www.myneta.info/LokSabha2024/index.php...,19-04-2024
1,2024,ANDHRA PRADESH,2,AMALAPURAM (SC),7,https://www.myneta.info/LokSabha2024/index.php...,13-05-2024
2,2024,ANDHRA PRADESH,2,ANAKAPALLE,5,https://www.myneta.info/LokSabha2024/index.php...,13-05-2024
3,2024,ANDHRA PRADESH,2,ANANTHAPUR,20,https://www.myneta.info/LokSabha2024/index.php...,13-05-2024
4,2024,ANDHRA PRADESH,2,ARAKU (ST),1,https://www.myneta.info/LokSabha2024/index.php...,13-05-2024


Scraping 2019...
2019: extracted 559 constituency links


,year,state_ut,state_id,const,const_id,link,election_date
0,2019,ANDAMAN & NICOBAR ISLANDS,62,ANDAMAN AND NICOBAR ISLANDS,431,https://www.myneta.info/LokSabha2019/index.php...,11-04-2019
1,2019,ANDHRA PRADESH,34,AMALAPURAM,438,https://www.myneta.info/LokSabha2019/index.php...,11-04-2019
2,2019,ANDHRA PRADESH,34,ANAKAPALLE,436,https://www.myneta.info/LokSabha2019/index.php...,11-04-2019
3,2019,ANDHRA PRADESH,34,ANANTAPUR,450,https://www.myneta.info/LokSabha2019/index.php...,11-04-2019
4,2019,ANDHRA PRADESH,34,ARAKU,432,https://www.myneta.info/LokSabha2019/index.php...,11-04-2019


Scraping 2014...
2014: extracted 574 constituency links


...

,year,state_ut,constituency_count
0,2014,ANDAMAN & NICOBAR ISLANDS,1
1,2014,ANDHRA PRADESH,25
2,2014,ARUNACHAL PRADESH,2
3,2014,ASSAM,15
4,2014,BIHAR,41
...,...,...,...
103,2024,TELANGANA,17
104,2024,TRIPURA,2
105,2024,UTTAR PRADESH,80
106,2024,UTTARAKHAND,5



Missing values:


,missing_count
year,0
state_ut,0
state_id,0
const,0
const_id,0
link,0
election_date,50



Target-state rows:


,year,state_ut,constituency_count
0,2014,BIHAR,41
1,2014,MAHARASHTRA,51
2,2014,TAMIL NADU,39
3,2014,UTTAR PRADESH,84
4,2019,BIHAR,42
5,2019,MAHARASHTRA,49
6,2019,TAMIL NADU,40
7,2019,UTTAR PRADESH,83
8,2024,BIHAR,40
9,2024,MAHARASHTRA,49


### Extracting the responses from each constituency URL

In [12]:
import pandas as pd
import requests
import re
import time
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = Path("../raw")
LINK_DIR = BASE_DIR / "lok_sabha_constituencies"
HTML_DIR = BASE_DIR / "lok_sabha_html"
HTML_DIR.mkdir(parents=True, exist_ok=True)

TARGET_STATES = [
    "TAMIL NADU",
    "UTTAR PRADESH",
    "BIHAR",
    "MAHARASHTRA",
]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36"
})

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

manifest_path = HTML_DIR / "html_manifest.csv"

if manifest_path.exists():
    old_manifest = pd.read_csv(manifest_path)
else:
    old_manifest = pd.DataFrame()

manifest_rows = []

for year in [2024, 2019, 2014]:
    link_path = LINK_DIR / f"constituency_links_{year}.csv"
    df = pd.read_csv(link_path)

    selected = df[
        df["state_ut"].fillna("").str.strip().str.upper().isin(TARGET_STATES)
    ]

    print(f"{year}: {len(selected)} constituency pages selected")

    for _, row in selected.iterrows():
        state = str(row["state_ut"]).strip()
        const = str(row["const"]).strip()
        const_id = str(row["const_id"])
        url = row["link"]

        safe_state = re.sub(r"[^A-Za-z0-9_-]+", "_", state)
        safe_const = re.sub(r"[^A-Za-z0-9_-]+", "_", const)
        folder = HTML_DIR / str(year) / safe_state
        folder.mkdir(parents=True, exist_ok=True)
        html_path = folder / f"{safe_const}_{const_id}.html"

        # Reuse an existing successful download.
        previous_ok = False
        if not old_manifest.empty and "status" in old_manifest.columns:
            previous = old_manifest[
                old_manifest["link"].eq(url)
                & old_manifest["status"].eq("ok")
            ]
            if len(previous) and html_path.exists():
                previous_ok = True

        if previous_ok:
            status = "ok"
            http_status = 200
            error = None
        else:
            try:
                response = session.get(url, timeout=(15, 60))
                response.raise_for_status()
                html_path.write_bytes(response.content)
                status = "ok"
                http_status = response.status_code
                error = None
            except requests.RequestException as e:
                status = "error"
                http_status = None
                error = str(e)

        manifest_rows.append({
            "year": year,
            "state_ut": state,
            "state_id": row["state_id"],
            "const": const,
            "const_id": const_id,
            "link": url,
            "election_date": row["election_date"],
            "html_path": str(html_path),
            "status": status,
            "http_status": http_status,
            "error": error,
        })

        time.sleep(0.4)

columns = [
    "year", "state_ut", "state_id", "const", "const_id", "link",
    "election_date", "html_path", "status", "http_status", "error"
]

manifest = pd.DataFrame(manifest_rows, columns=columns)
manifest.to_csv(manifest_path, index=False)

print("\nPages selected:", len(manifest))
print("Downloaded:", manifest["status"].eq("ok").sum())
print("Failed:", manifest["status"].eq("error").sum())

display(
    manifest.groupby(["year", "state_ut", "status"], dropna=False)
    .size().reset_index(name="count")
)

2024: 208 constituency pages selected
2019: 214 constituency pages selected
2014: 215 constituency pages selected

Pages selected: 637
Downloaded: 637
Failed: 0


,year,state_ut,status,count
0,2014,BIHAR,ok,41
1,2014,MAHARASHTRA,ok,51
2,2014,TAMIL NADU,ok,39
3,2014,UTTAR PRADESH,ok,84
4,2019,BIHAR,ok,42
5,2019,MAHARASHTRA,ok,49
6,2019,TAMIL NADU,ok,40
7,2019,UTTAR PRADESH,ok,83
8,2024,BIHAR,ok,40
9,2024,MAHARASHTRA,ok,49


### Extracting the data into a pandas data frame

In [13]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

HTML_DIR = Path("../raw/lok_sabha_html")
OUT_DIR = Path("../raw/lok_sabha_candidate_dfs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = HTML_DIR / "html_manifest.csv"

if not manifest_path.exists():
    raise FileNotFoundError("Run Phase 2 first.")

manifest = pd.read_csv(manifest_path)
candidate_rows = []

def header_key(text):
    return re.sub(r"[^a-z]", "", text.lower())

def find_column(headers, possible_names):
    for i, header in enumerate(headers):
        key = header_key(header)
        if any(name in key for name in possible_names):
            return i
    return None

def get_cell_value(cells, index):
    if index is None or index >= len(cells):
        return None
    value = cells[index].get_text(" ", strip=True)
    return value if value else None

for _, page in manifest[manifest["status"].eq("ok")].iterrows():
    html_path = Path(page["html_path"])

    if not html_path.exists():
        print("Missing HTML:", html_path)
        continue

    soup = BeautifulSoup(html_path.read_bytes(), "html.parser")

    for table in soup.find_all("table"):
        header_row = table.find("tr")
        if header_row is None:
            continue

        headers = [
            cell.get_text(" ", strip=True)
            for cell in header_row.find_all(["th", "td"], recursive=False)
        ]

        candidate_col = find_column(headers, ["candidate"])
        party_col = find_column(headers, ["party"])
        criminal_col = find_column(headers, ["criminalcases", "criminalcase"])
        education_col = find_column(headers, ["education"])
        age_col = find_column(headers, ["age"])
        assets_col = find_column(headers, ["totalassets", "assets"])
        liabilities_col = find_column(headers, ["liabilities"])

        # Skip tables that are not candidate-result tables.
        if candidate_col is None or assets_col is None:
            continue

        for tr in table.find_all("tr")[1:]:
            cells = tr.find_all(["td", "th"], recursive=False)

            if len(cells) <= max(candidate_col, assets_col):
                continue

            candidate_cell = cells[candidate_col]
            candidate_anchor = candidate_cell.find(
                "a", href=re.compile(r"candidate\.php", re.I)
            )

            if candidate_anchor is None:
                continue

            candidate_name = candidate_anchor.get_text(" ", strip=True)
            candidate_url = urljoin(page["link"], candidate_anchor["href"])
            candidate_id = parse_qs(
                urlparse(candidate_url).query
            ).get("candidate_id", [None])[0]

            candidate_text = candidate_cell.get_text(" ", strip=True)
            is_winner = bool(
                re.search(r"\bWinner\b", candidate_text, flags=re.I)
            )

            asset_cell = cells[assets_col]
            asset_link = None

            # Preserve the URL if assets are represented by an image.
            for img in asset_cell.find_all("img"):
                src = img.get("src") or img.get("data-src")
                if src:
                    asset_link = urljoin(page["link"], src)
                    break

            candidate_rows.append({
                "year": page["year"],
                "state": page["state_ut"],
                "state_id": page["state_id"],
                "const": page["const"],
                "const_id": page["const_id"],
                "candidate": candidate_name,
                "candidate_id": candidate_id,
                "candidate_url": candidate_url,
                "is_winner": is_winner,
                "party": get_cell_value(cells, party_col),
                "criminal_cases": get_cell_value(cells, criminal_col),
                "education": get_cell_value(cells, education_col),
                "age": get_cell_value(cells, age_col),
                "total_assets": get_cell_value(cells, assets_col),
                "asset_link": asset_link,
                "liabilities": get_cell_value(cells, liabilities_col),
                "constituency_url": page["link"],
                "html_path": str(html_path),
            })

columns = [
    "year", "state", "state_id", "const", "const_id",
    "candidate", "candidate_id", "candidate_url", "is_winner",
    "party", "criminal_cases", "education", "age",
    "total_assets", "asset_link", "liabilities",
    "constituency_url", "html_path"
]

candidates = pd.DataFrame(candidate_rows, columns=columns)

if candidates.empty:
    print("No candidate rows found. Check the HTML table structure.")
else:
    candidates.to_csv(OUT_DIR / "candidates_all_years.csv", index=False)

    for (year, state), group in candidates.groupby(
        ["year", "state"], dropna=False
    ):
        safe_state = re.sub(r"[^A-Za-z0-9_-]+", "_", str(state))
        folder = OUT_DIR / str(year) / safe_state
        folder.mkdir(parents=True, exist_ok=True)
        group.to_csv(folder / "candidates.csv", index=False)

    print("Total candidate rows:", len(candidates))
    print("Winners:", candidates["is_winner"].sum())
    print("Asset image links:", candidates["asset_link"].notna().sum())

    display(candidates.head(10))

    print("\nCandidate counts by year and state:")
    display(
        candidates.groupby(["year", "state"], dropna=False)
        .size().reset_index(name="candidate_count")
    )

Total candidate rows: 738
Winners: 634
Asset image links: 233


,year,state,state_id,const,const_id,candidate,candidate_id,candidate_url,is_winner,party,criminal_cases,education,age,total_assets,asset_link,liabilities,constituency_url,html_path
0,2024,BIHAR,5,ARARIA,56,Pradeep Kumar Singh,3627,https://www.myneta.info/LokSabha2024/candidate...,True,BJP,4,10th Pass,58,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
1,2024,BIHAR,5,ARARIA,56,Akhilesh Kumar,4040,https://www.myneta.info/LokSabha2024/candidate...,False,IND,0,Doctorate,47,"Rs 2,47,11,288 ~ 2 Crore+",NaN,"Rs 3,05,374 ~ 3 Lacs+",https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
2,2024,BIHAR,5,ARARIA,56,Md Ghousul Azam,4773,https://www.myneta.info/LokSabha2024/candidate...,False,BSP,0,10th Pass,38,"Rs 1,60,50,000 ~ 1 Crore+",NaN,"Rs 14,00,000 ~ 14 Lacs+",https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
3,2024,BIHAR,5,ARARIA,56,Md. Ismail,4107,https://www.myneta.info/LokSabha2024/candidate...,False,Bharatiya Momin Front,0,Graduate Professional,56,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
4,2024,BIHAR,5,ARARIA,56,Md. Mobinul Haque,4772,https://www.myneta.info/LokSabha2024/candidate...,False,IND,0,Post Graduate,51,"Rs 34,05,000 ~ 34 Lacs+",NaN,Rs 0 ~,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
5,2024,BIHAR,5,ARARIA,56,Mushtak Alam,4039,https://www.myneta.info/LokSabha2024/candidate...,False,IND,2,5th Pass,49,"Rs 7,28,90,900 ~ 7 Crore+",NaN,"Rs 3,50,000 ~ 3 Lacs+",https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
6,2024,BIHAR,5,ARARIA,56,Shahnawaz,4038,https://www.myneta.info/LokSabha2024/candidate...,False,RJD,1,Graduate Professional,42,"Rs 3,33,05,135 ~ 3 Crore+",NaN,Rs 0 ~,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
7,2024,BIHAR,5,ARARIA,56,Shatrughan Prasad Suman,4774,https://www.myneta.info/LokSabha2024/candidate...,False,IND,0,Graduate Professional,37,"Rs 89,49,490 ~ 89 Lacs+",NaN,Rs 0 ~,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARARIA_56.html
8,2024,BIHAR,5,ARRAH,82,Sudama Prasad,9539,https://www.myneta.info/LokSabha2024/candidate...,True,CPI(ML)(L),3,Graduate,56,NaN,https://myneta.info/image_v2.php?myneta_folder...,NaN,https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARRAH_82.html
9,2024,BIHAR,5,ARRAH,82,Ashok Tiwary,9544,https://www.myneta.info/LokSabha2024/candidate...,False,IND,0,Graduate,46,"Rs 45,46,740 ~ 45 Lacs+",NaN,"Rs 10,33,200 ~ 10 Lacs+",https://www.myneta.info/LokSabha2024/index.php...,..\raw\lok_sabha_html\2024\BIHAR\ARRAH_82.html



Candidate counts by year and state:


,year,state,candidate_count
0,2014,BIHAR,41
1,2014,MAHARASHTRA,50
2,2014,TAMIL NADU,39
3,2014,UTTAR PRADESH,83
4,2019,BIHAR,42
5,2019,MAHARASHTRA,49
6,2019,TAMIL NADU,40
7,2019,UTTAR PRADESH,82
8,2024,BIHAR,144
9,2024,MAHARASHTRA,49
